# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [1]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    with open(filename, "a") as file:
        file.write(f"[{timestamp}] {text}\n")

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    try:
        with open(filename, "r") as file:
            for line in file:
                print(line.strip())
    except FileNotFoundError:
        print("Journal not found. No entries yet.")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    try:
        with open(filename, "r") as file:
            lines = file.readlines()
    except FileNotFoundError:
        return []
    
    clean_lines = [line.strip() for line in lines]
    return [line for line in clean_lines if keyword.lower() in line.lower()]

def count_entries(filename):
    """Count and return the number of journal entries."""
    try:
        with open(filename, "r") as file:
            lines = file.readlines()
    except FileNotFoundError:
        return 0

    return len([line for line in lines if line.strip() != ""])

def run_journal():
    """Menu-driven journal interface."""
    while True:
        print("\n[A]dd  [R]ead  [S]earch  [C]ount  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            text = input("Write your entry: ")
            add_entry(JOURNAL_FILE, text)
            print("Entry added.")
        elif choice == "r":
            read_journal(JOURNAL_FILE)
        elif choice == "s":
            keyword = input("Keyword: ")
            results = search_journal(JOURNAL_FILE, keyword)
            print(f"Found {len(results)} result(s):")
            for line in results:
                print(line)
        elif choice == "c":
            print(f"Total entries: {count_entries(JOURNAL_FILE)}")
        elif choice == "q":
            print("Goodbye!")
            break
        else:
            print("Invalid option.")


# --- QUICK TEST (non-interactive) ---
add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
read_journal(JOURNAL_FILE)
print("Search results:", search_journal(JOURNAL_FILE, "python"))
print("Total entries:", count_entries(JOURNAL_FILE))

run_journal()  # Uncomment to test the interactive menu

[2026-10-22 03:32] Started learning about file I/O today!
[2026-10-22 03:32] Python makes reading files so easy.
Search results: ['[2026-10-22 03:32] Python makes reading files so easy.']
Total entries: 2

[A]dd  [R]ead  [S]earch  [C]ount  [Q]uit
Goodbye!


---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [2]:
import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from a JSON file. Return [] if missing or corrupted."""
    try:
        with open(filename, "r") as file:
            return json.load(file)
    except FileNotFoundError:
        print("No existing tasks found. Starting fresh.")
        return []
    except json.JSONDecodeError:
        print("Error: File is corrupted. Starting fresh.")
        return []

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    with open(filename, "w") as file:
        json.dump(tasks, file, indent=2)

def add_task(tasks):
    """Prompt for title and priority, append new task."""
    title = input("Task title: ").strip()
    while title == "":
        print("Title cannot be empty.")
        title = input("Task title: ").strip()

    priority = input("Priority (high/medium/low): ").strip().lower()
    while priority not in ["high", "medium", "low"]:
        print("Invalid priority. Choose high, medium or low.")
        priority = input("Priority (high/medium/low): ").strip().lower()

    tasks.append({"title": title, "priority": priority, "done": False})
    print(f"Task '{title}' added.")

def list_tasks(tasks):
    """Display all tasks in a formatted table."""
    if not tasks:
        print("No tasks yet.")
        return

    print(f"{'#':<4}{'Status':<8}{'Title':<30}{'Priority':<10}")
    print("-" * 52)
    for i, task in enumerate(tasks):
        status = "[x]" if task["done"] else "[ ]"
        print(f"{i:<4}{status:<8}{task['title']:<30}{task['priority']:<10}")

def mark_done(tasks):
    """Ask for a task index and mark it as done."""
    if not tasks:
        print("No tasks to mark.")
        return
    list_tasks(tasks)

    try:
        index = int(input("Task number: "))
    except ValueError:
        print("Please enter a number.")
        return

    if index < 0 or index >= len(tasks):
        print("That task number does not exist.")
        return

    tasks[index]["done"] = True
    print(f"Task '{tasks[index]['title']}' marked as done.")

def delete_task(tasks):
    """Ask for a task index and remove the task."""
    if not tasks:
        print("No tasks to delete.")
        return
    list_tasks(tasks)

    try:
        index = int(input("Task number: "))
    except ValueError:
        print("Please enter a number.")
        return

    if index < 0 or index >= len(tasks):
        print("That task number does not exist.")
        return

    removed = tasks.pop(index)
    print(f"Task '{removed['title']}' deleted.")

def run_todo():
    """Main menu loop with auto-save."""
    tasks = load_tasks(TODO_FILE)

    while True:
        print("\n[A]dd  [L]ist  [D]one  [R]emove  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "l":
            list_tasks(tasks)
        elif choice == "d":
            mark_done(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "r":
            delete_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "q":
            print("Goodbye!")
            break
        else:
            print("Invalid option.")


# --- QUICK TEST (non-interactive) ---
tasks = load_tasks(TODO_FILE)
tasks.append({"title": "Test task", "priority": "high", "done": False})
save_tasks(TODO_FILE, tasks)
tasks = load_tasks(TODO_FILE)
list_tasks(tasks)

run_todo()  # Uncomment to test the interactive menu

No existing tasks found. Starting fresh.
#   Status  Title                         Priority  
----------------------------------------------------
0   [ ]     Test task                     high      

[A]dd  [L]ist  [D]one  [R]emove  [Q]uit
Goodbye!


---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [4]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

✅ expenses.csv created successfully.


In [5]:
import csv

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    expenses = []
    try:
        with open(filename, "r") as file:
            reader = csv.DictReader(file)
            for row in reader:
                # Validate category
                if row["category"].strip() == "":
                    print(f"Skipped (empty category): {row['description']}")
                    continue
                # Validate amount
                try:
                    amount = float(row["amount"])
                except ValueError:
                    print(f"Skipped (invalid amount '{row['amount']}'): {row['description']}")
                    continue

                row["amount"] = amount
                expenses.append(row)
    except FileNotFoundError:
        print(f"File '{filename}' not found.")
        return []

    return expenses

def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    totals = {}
    for expense in expenses:
        category = expense["category"]
        totals[category] = totals.get(category, 0) + expense["amount"]
    return totals

def most_expensive(expenses):
    """Return the expense dict with the highest amount."""
    if not expenses:
        return None
    return max(expenses, key=lambda e: e["amount"])

def print_report(expenses):
    """Print a formatted expense summary report."""
    if not expenses:
        print("No valid expenses to report.")
        return

    total = sum(e["amount"] for e in expenses)
    print("\n===== EXPENSE REPORT =====")
    print(f"Valid entries: {len(expenses)}")
    print(f"Total spent: {total:.2f} €")

    print("\nBy category:")
    totals = total_by_category(expenses)
    for category, amount in sorted(totals.items(), key=lambda item: item[1], reverse=True):
        print(f"  {category:<15} {amount:>8.2f} €")

    top = most_expensive(expenses)
    print(f"\nMost expensive: {top['description']} ({top['category']}) - {top['amount']:.2f} € on {top['date']}")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)

Skipped (invalid amount 'bad_data'): Gas station
Skipped (empty category): Gym membership
Skipped (invalid amount 'not_a_number'): Water bill

===== EXPENSE REPORT =====
Valid entries: 13
Total spent: 510.13 €

By category:
  Food              162.75 €
  Entertainment     114.99 €
  Utilities          89.00 €
  Education          72.49 €
  Transport          58.50 €
  Health             12.40 €

Most expensive: Electricity bill (Utilities) - 89.00 € on 2024-01-13


---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [6]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

✅ All test files created successfully.


In [7]:
import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}

    try:
        with open(filename, "r") as file:
            data = json.load(file)
    except FileNotFoundError:
        report["errors"].append("File not found.")
        return report
    except json.JSONDecodeError:
        report["errors"].append("File contains invalid JSON.")
        return report

    if not isinstance(data, list):
        report["errors"].append("Root element is not a list.")
        return report

    for i, item in enumerate(data):
        if not isinstance(item, dict):
            report["invalid"] += 1
            report["errors"].append(f"Item {i}: not a dictionary.")
            continue

        missing = [key for key in required_keys if key not in item]
        if missing:
            report["invalid"] += 1
            report["errors"].append(f"Item {i}: missing keys {missing}.")
            continue

        report["valid"] += 1

    return report

def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}

    try:
        with open(filename, "r") as file:
            reader = csv.reader(file)

            try:
                header = next(reader)
            except StopIteration:
                report["errors"].append("File is empty.")
                return report

            if len(header) != expected_columns:
                report["errors"].append(
                    f"Header has {len(header)} columns, expected {expected_columns}."
                )
                return report

            for i, row in enumerate(reader, start=2):
                if len(row) != expected_columns:
                    report["invalid"] += 1
                    report["errors"].append(
                        f"Row {i}: has {len(row)} fields, expected {expected_columns}."
                    )
                else:
                    report["valid"] += 1

    except FileNotFoundError:
        report["errors"].append("File not found.")

    return report

def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    print(f"Validation report for '{filename}'")
    print(f"  Valid records:   {report['valid']}")
    print(f"  Invalid records: {report['invalid']}")

    if report["errors"]:
        print("  Errors:")
        for error in report["errors"]:
            print(f"    - {error}")
    else:
        print("  No errors found")


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")

Validation report for 'good_data.json'
  Valid records:   3
  Invalid records: 0
  No errors found
Validation report for 'bad_data.json'
  Valid records:   2
  Invalid records: 2
  Errors:
    - Item 1: missing keys ['email'].
    - Item 2: not a dictionary.
Validation report for 'corrupted.json'
  Valid records:   0
  Invalid records: 0
  Errors:
    - File contains invalid JSON.
Validation report for 'good_data.csv'
  Valid records:   2
  Invalid records: 0
  No errors found
Validation report for 'bad_data.csv'
  Valid records:   2
  Invalid records: 2
  Errors:
    - Row 3: has 2 fields, expected 3.
    - Row 4: has 4 fields, expected 3.


---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [8]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

✅ students.json created.


In [9]:
import json
import csv

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    try:
        with open(json_filename, "r") as file:
            data = json.load(file)
    except FileNotFoundError:
        print(f"Error: '{json_filename}' not found.")
        return
    except json.JSONDecodeError:
        print(f"Error: '{json_filename}' contains invalid JSON.")
        return

    if not isinstance(data, list) or len(data) == 0:
        print(f"Error: '{json_filename}' is empty or not a list.")
        return
    if not all(isinstance(item, dict) for item in data):
        print(f"Error: '{json_filename}' must contain only dictionaries.")
        return

    fieldnames = list(data[0].keys())
    try:
        with open(csv_filename, "w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
    except ValueError:
        print("Error: some records have keys that don't match the first record.")
        return

    print(f"Converted {json_filename} → {csv_filename} ({len(data)} records).")

def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    try:
        with open(csv_filename, "r") as file:
            reader = csv.DictReader(file)
            data = list(reader)
    except FileNotFoundError:
        print(f"Error: '{csv_filename}' not found.")
        return

    if len(data) == 0:
        print(f"Error: '{csv_filename}' has no data rows.")
        return

    with open(json_filename, "w") as file:
        json.dump(data, file, indent=2)

    print(f"Converted {csv_filename} → {json_filename} ({len(data)} records).")

def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only records where record[filter_key] == filter_value to CSV."""
    try:
        with open(json_filename, "r") as file:
            data = json.load(file)
    except FileNotFoundError:
        print(f"Error: '{json_filename}' not found.")
        return
    except json.JSONDecodeError:
        print(f"Error: '{json_filename}' contains invalid JSON.")
        return

    if not isinstance(data, list):
        print(f"Error: '{json_filename}' is not a list.")
        return

    filtered = [
        record for record in data
        if isinstance(record, dict) and record.get(filter_key) == filter_value
    ]

    if len(filtered) == 0:
        print(f"No records found where {filter_key} == {filter_value!r}. Nothing exported.")
        return

    with open(csv_filename, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(filtered[0].keys()))
        writer.writeheader()
        writer.writerows(filtered)

    print(f"Exported {len(filtered)} record(s) where {filter_key} == {filter_value!r} → {csv_filename}.")


# --- TEST ---
# JSON → CSV
json_to_csv("students.json", "students_export.csv")

# CSV → JSON (round-trip test)
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

# Verify round-trip
with open("students_roundtrip.json", "r") as f:
    roundtrip = json.load(f)
print(f"\nRound-trip data ({len(roundtrip)} records):")
for r in roundtrip:
    print(f"  {r}")

Converted students.json → students_export.csv (5 records).
Converted students_export.csv → students_roundtrip.json (5 records).
Exported 2 record(s) where city == 'Madrid' → madrid_students.csv.

Round-trip data (5 records):
  {'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
  {'name': 'Bob', 'age': '25', 'city': 'London', 'grade': 'B'}
  {'name': 'Charlie', 'age': '30', 'city': 'Paris', 'grade': 'A'}
  {'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}
  {'name': 'Eve', 'age': '23', 'city': 'London', 'grade': 'A'}


---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [10]:
import json
import csv

DB_FILE = "students_db.json"

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    try:
        with open(filename, "r") as file:
            return json.load(file)
    except FileNotFoundError:
        return []
    except json.JSONDecodeError:
        print("Warning: database file is corrupted. Starting with an empty database.")
        return []

def save_db(filename, students):
    """Save student records to JSON."""
    with open(filename, "w") as file:
        json.dump(students, file, indent=2)

# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    if str(name).strip() == "":
        return False, "Name cannot be empty."

    try:
        age = int(age)
    except ValueError:
        return False, f"Age '{age}' is not an integer."
    if age <= 0:
        return False, "Age must be a positive integer."

    if "@" not in str(email):
        return False, f"Email '{email}' must contain '@'."

    if str(grade).strip().upper() not in ["A", "B", "C", "D", "F"]:
        return False, f"Grade '{grade}' must be one of A, B, C, D, F."

    return True, ""

# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    name = input("Name: ").strip()
    age = input("Age: ").strip()
    email = input("Email: ").strip()
    grade = input("Grade (A/B/C/D/F): ").strip().upper()

    is_valid, error = validate_student(name, age, email, grade)
    if not is_valid:
        print(f"Student not added: {error}")
        return

    students.append({"name": name, "age": int(age), "email": email, "grade": grade})
    print(f"Student '{name}' added.")

def list_students(students):
    """Display all students in a formatted table."""
    if not students:
        print("No students in the database.")
        return

    print(f"{'Name':<15}{'Age':<6}{'Email':<25}{'Grade':<6}")
    print("-" * 52)
    for s in students:
        print(f"{s['name']:<15}{s['age']:<6}{s['email']:<25}{s['grade']:<6}")

def search_student(students):
    """Search by name (case-insensitive partial match)."""
    term = input("Search name: ").strip().lower()
    results = [s for s in students if term in s["name"].lower()]

    if not results:
        print("No students found.")
        return
    print(f"Found {len(results)} student(s):")
    list_students(results)

def update_student(students):
    """Update a student's grade or email by name."""
    name = input("Name of the student to update: ").strip()

    student = None
    for s in students:
        if s["name"].lower() == name.lower():
            student = s
            break

    if student is None:
        print(f"No student named '{name}'.")
        return

    field = input("Update [G]rade or [E]mail? ").strip().lower()
    if field == "g":
        new_grade = input("New grade (A/B/C/D/F): ").strip().upper()
        if new_grade not in ["A", "B", "C", "D", "F"]:
            print("Invalid grade. Nothing changed.")
            return
        student["grade"] = new_grade
        print(f"Grade updated to {new_grade}.")
    elif field == "e":
        new_email = input("New email: ").strip()
        if "@" not in new_email:
            print("Invalid email. Nothing changed.")
            return
        student["email"] = new_email
        print(f"Email updated to {new_email}.")
    else:
        print("Invalid option. Nothing changed.")

def delete_student(students):
    """Remove a student by name."""
    name = input("Name of the student to delete: ").strip()

    student = None
    for s in students:
        if s["name"].lower() == name.lower():
            student = s
            break

    if student is None:
        print(f"No student named '{name}'.")
        return

    confirm = input(f"Are you sure you want to delete '{student['name']}'? (y/n): ").strip().lower()
    if confirm == "y":
        students.remove(student)
        print(f"Student '{student['name']}' deleted.")
    else:
        print("Deletion cancelled.")

# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    try:
        with open(csv_filename, "r") as file:
            reader = csv.DictReader(file)
            rows = list(reader)
    except FileNotFoundError:
        print(f"File '{csv_filename}' not found.")
        return

    added = 0
    skipped = 0
    for i, row in enumerate(rows, start=2):   # row 1 is the header
        is_valid, error = validate_student(row["name"], row["age"], row["email"], row["grade"])
        if not is_valid:
            print(f"  Skipped row {i} ({row['name']}): {error}")
            skipped += 1
            continue

        students.append({
            "name": row["name"].strip(),
            "age": int(row["age"]),
            "email": row["email"].strip(),
            "grade": row["grade"].strip().upper(),
        })
        added += 1

    print(f"Import finished: {added} added, {skipped} skipped.")

def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    if not students:
        print("No students to export.")
        return

    with open(csv_filename, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["name", "age", "email", "grade"])
        writer.writeheader()
        writer.writerows(students)
    print(f"Exported {len(students)} student(s) to '{csv_filename}'.")

# --- MAIN MENU ---

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

run_student_db()  # Uncomment to test

Loaded 0 student(s) from database.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Database saved. Goodbye!


In [11]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")

import_students.csv created (3 valid, 3 invalid rows).
